# Lazypredict

In [1]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [2]:
df = pd.read_excel("../Data Final/Final_PM14-Facial.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Start_Time               177 non-null    datetime64[us]
 1   End_Time                 177 non-null    datetime64[us]
 2   Data_Count               177 non-null    int64         
 3   Mean_Creping             177 non-null    float64       
 4   Mean_Yankee Speed        177 non-null    float64       
 5   Mean_Pope Reel Speed     177 non-null    float64       
 6   Mean_Yankee Pressure     177 non-null    float64       
 7   Mean_Stock Flow          177 non-null    float64       
 8   Mean_Stock Consistency   177 non-null    float64       
 9   Mean_Flow Coating        177 non-null    float64       
 10  Mean_Flow Release        177 non-null    float64       
 11  Mean_Jet Wire Ratio      177 non-null    float64       
 12  Mean_Load KWH Refiner    177 non-null    float6

In [3]:
# Data Preparation

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    'Mean_Yankee Pressure',
    'Mean_Creping',
    '% NBKP',
    'Mean_Load KWH Refiner',
    'Mean_Jet Wire Ratio',
    'GSM',
    'coating_release_ratio',
    'pseudo_mass'
]
X = df[features]

# Y Variables
y = df['MDT']

In [4]:
X.tail()

,Mean_Yankee Pressure,Mean_Creping,% NBKP,Mean_Load KWH Refiner,Mean_Jet Wire Ratio,GSM,coating_release_ratio,pseudo_mass
172,7.398000,26.024178,20.071873,193.279860,0.925,14.5,0.995310,3.611408
173,7.397951,26.008332,20.071873,195.859918,0.925,14.5,0.995311,3.621878
174,7.398288,25.991434,20.071873,196.420904,0.925,14.5,0.995311,3.624618
175,7.397400,26.000439,20.071873,193.188673,0.925,14.5,0.995308,3.636918
176,7.398333,26.000551,20.071873,194.652458,0.925,14.5,0.995309,3.638424


In [5]:
y.head()

0    279
1    332
2    228
3    268
4    317
Name: MDT, dtype: int64

In [6]:
print("Min Pseudo Mass: ", X['pseudo_mass'].min())
print("Max Pseudo Mass: ", X['pseudo_mass'].max())

Min Pseudo Mass:  2.860909550442946
Max Pseudo Mass:  3.672565186563839


In [6]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [8]:
# Hitung MAPE untuk tiap model
mape_scores = {}

for model_name, y_pred in predictions.items():
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mape_scores[model_name] = mape

# Tambahkan ke tabel hasil
models["MAPE"] = pd.Series(mape_scores)

# Urutkan (semakin kecil semakin baik)
models = models.sort_values(by="MAPE")

In [9]:
print(models)

                               Adjusted R-Squared  R-Squared        RMSE  \
Model                                                                      
ExtraTreesRegressor                      0.129503   0.328474   47.710488   
ElasticNet                               0.078923   0.289455   49.077026   
TweedieRegressor                         0.070101   0.282650   49.311492   
GammaRegressor                           0.067794   0.280869   49.372642   
GradientBoostingRegressor                0.033209   0.254190   50.280165   
RandomForestRegressor                    0.026209   0.248790   50.461862   
SVR                                      0.013236   0.238782   50.796883   
OrthogonalMatchingPursuit                0.013119   0.238692   50.799874   
KNeighborsRegressor                      0.009023   0.235532   50.905195   
ElasticNetCV                             0.006800   0.233817   50.962263   
Lasso                                   -0.015994   0.216233   51.543746   
LassoLars   

# Regressor

In [10]:
# Import Libraries
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# Model Dasar
model = ExtraTreesRegressor(random_state=42,n_jobs=1)

# Grid Search Hyper Parameter
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 8, 10, 15],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 0.5, 0.7],
    'bootstrap': [True, False]
}

# K-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GGrid Search CV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)

# Training + Tuning
grid_search.fit(X, y)

# Model Terbaik
best_model = grid_search.best_estimator_
print("Best Parameters:")
print(grid_search.best_params_)

# Prrediksi
y_pred = best_model.predict(X)

# Matrik Evaluasi
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
epsilon = 1e-8
mape = np.mean(np.abs((y - y_pred) / (y + epsilon))) * 100

# Hasil
print("\n===== HASIL MODEL TERBAIK =====")
print(f"R2    : {r2:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAE   : {mae:.4f}")
print(f"MAPE  : {mape:.2f}%")

Fitting 5 folds for each of 1152 candidates, totalling 5760 fits
Best Parameters:
{'bootstrap': False, 'max_depth': 10, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 50}

===== HASIL MODEL TERBAIK =====
R2    : 0.9056
RMSE  : 22.1480
MAE   : 16.3901
MAPE  : 4.35%


### Pickle Files

In [11]:
import joblib

In [ ]:
#joblib.dump(best_model, 'model_PM14-Facial.pkl')

['model_PM14-Facial.pkl']

In [13]:
features_pkl = X.columns.tolist()
#joblib.dump(features_pkl, 'features_PM14-Facial.pkl')

### Features Importance

In [14]:
# Feature Importance
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": features, 
    "Importance": model.feature_importances_
})

# Urutkan dari terbesar
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)


# Plot
import matplotlib.pyplot as plt
plt.figure()
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Extra Tree")

plt.tight_layout()
plt.show()

NotFittedError: This ExtraTreesRegressor instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

### SHAP Analysis

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(list(X_train.columns))
print(list(X_test.columns))

In [ ]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
shap.plots.scatter(shap_values[:, "Mean_Load KWH Refiner"])

In [ ]:
shap.plots.scatter(shap_values[:, "pseudo_mass"])

In [ ]:
shap_values = explainer.shap_values(np.array(X_test))
shap.initjs()
i = 0  # index data

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)